In [1]:
import os
from collections import defaultdict
import pandas as pd
import numpy as np
from Bio import SeqIO
import random

In [2]:
## define path
basedir = "/Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx"

# output directory
path_out = f"{basedir}/output/251111_3921_samples_external_final_test/"

# input files
# blastx output directory
path_in = f"{basedir}/data/250821_3883samples_external_downloaded251110/250821_3883samples_external/blastout_all_protein/"
# fasta file
path_in_fasta = f"{basedir}/data/250821_3883samples_external_downloaded251110/250821_3883samples_external/contig_viral_hit/"
# sample info table
sradata_path = f"{basedir}/data/data_external_251110/sample_list_cuttlefish_minia_diamond_add_positive0to2_blastx_all.txt"
srameta_path = f"{basedir}/data/data_external_251110/Aves_Mam_mbio_metadata_ISG_nologan3942_ML_251110.txt"
# blacklist virus
blacklist_path = f"{basedir}/data/black_list_TN_ISG_low_new_format.txt"
# blastn output
path_in_blastn = f"{basedir}/data/251111_external/virus_hit_all_contigs_external.txt"

if not os.path.exists(path_out):
    os.mkdir(path_out)
print("saving files:", path_out)

saving files: /Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/251111_3921_samples_external_final_test/


In [3]:
def get_all_taxids(root_dir, takonkit_path):
    """
    extracts taxids from all samples
    """
    # initialize set
    taxids = set()

    # read blastx result file
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # to pass ".DS_store"
                continue

            # full path to each blast output file
            full_path = os.path.join(dirpath, filename)

            # read file
            col_name = [
                "qseqid", "sseqid", "pident", "length", "mismatch",
                "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
                "bitscore", "qframe", "staxids", "stitle", "qlen", 
                "slen", "qcovhsp", "scovhsp"
            ]

            if os.path.getsize(full_path) == 0:
                print("file size 0 ")
                continue

            one_sample = pd.read_table(full_path, header=None, dtype={13:str})  # staxids as string
            one_sample.columns = col_name

            # check if bitsocre exist for all queries
            if one_sample['bitscore'].isna().sum() > 0:
                print("NaN", filename)
            if (one_sample['bitscore'].dtype != np.float64) and (one_sample['bitscore'].dtype != int):
                print("dtype", filename, one_sample['bitscore'].dtype)
                print(one_sample['bitscore'])

            # sort by bitscore
            one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

            # delete duplicates
            one_sample_uniq = one_sample_sored.groupby("qseqid").first()

            # get taxid
            for line_taxid in one_sample_uniq["staxids"].to_list():
                if (line_taxid is None) or (line_taxid!=line_taxid):
                    continue
                raw_taxids = line_taxid.split(";")
                for i in range(len(raw_taxids)):
                    taxids.add(raw_taxids[i])

    # write taxid per line
    with open(takonkit_path, "w") as f:
        for taxid in sorted(list(taxids)):
            f.write(taxid + "\n")
    print(f"{len(taxids)} taxids in total")

    return taxids


def get_viral_hits(input_file, out_dir, lineage_info, blacklst):
    """
    saves only virus contigs based on taxid in the input table.
    """
    # num taxids with multiple groups
    num_ambiguous_hit = 0

    # confusion matrix group
    cm_group = 'external'

    # output file path
    output_folder = out_dir + "/taxonomy_output/"
    if not(os.path.exists(output_folder)):
        os.mkdir(output_folder)

    output_all_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.taxonomy.txt"
    output_viral_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.txt"
    output_viral_black_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.black.txt"

    # read file
    col_name = [
        "qseqid", "sseqid", "pident", "length", "mismatch",
        "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
        "bitscore", "qframe", "staxids", "stitle", "qlen", "slen",
        "qcovhsp", "scovhsp"
    ]

    one_sample = pd.read_table(input_file, header=None, dtype={13:str})
    one_sample.columns = col_name

    # add sample name
    one_sample["file_name"] = input_file.split("/")[-1].split(".")[0].split("_")[0]

    # sort by bitscore
    one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

    # delete duplicates
    one_sample_uniq = one_sample_sored.groupby("qseqid").first()

    # number of all contigs
    num_all_hits = len(set(one_sample_uniq.index.to_list()))

    # add taxonomy info
    one_sample_tax = one_sample_uniq.copy()
    one_sample_tax["staxids_lst"] = one_sample_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
    one_sample_tax["name_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
    one_sample_tax["group_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

    # add family and genus for the first taxid
    for idx in one_sample_tax.index:
        groups = one_sample_tax.loc[idx, "group_lst"]
        names = one_sample_tax.loc[idx, "name_lst"]
        if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
            num_ambiguous_hit += 1

    one_sample_tax["taxid_blastx"] = one_sample_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  # added on 2025/09/01
    one_sample_tax["family"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "family"] if((x is not None) and (x==x)) else x)
    one_sample_tax["genus"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "genus"] if((x is not None) and (x==x)) else x)
    one_sample_tax["species"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "species"] if((x is not None) and (x==x)) else x)

    # add tag if list has virus - virus = yes when all classification are virus
    one_sample_tax["is_virus"] = one_sample_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

    # extract only viral hit
    one_sample_virus = one_sample_tax.copy()
    one_sample_virus = one_sample_virus[one_sample_virus["is_virus"] == 1]

    # number of viral contigs
    num_viral_hits = len(set(one_sample_virus.index.to_list()))

    # read black list
    blackvirus = []
    with open(blacklst, "r") as f_in:
        for line in f_in:
            blackvirus.append(line.strip())

    # mask virus in black list
    one_sample_black_removed = one_sample_virus.copy()
    one_sample_black_removed = one_sample_black_removed[~one_sample_black_removed["family"].isin(blackvirus)]

    # number of viral contigs after filtering out black list virus
    num_viral_hits_after_filter = len(set(one_sample_black_removed.index.to_list()))

    # save
    one_sample_tax.to_csv(output_all_path, sep="\t")
    one_sample_virus.to_csv(output_viral_path, sep="\t")
    one_sample_black_removed.to_csv(output_viral_black_path, sep="\t")

    return one_sample_black_removed, num_all_hits, num_viral_hits, num_viral_hits_after_filter, num_ambiguous_hit


def count_viral_contigs(path_in: str, path_out, lineage_info, blacklist_path, infection_th=1):
    """
    counts virus contigs.
    """
    # initalize dictionary
    cm_hits = defaultdict(list)
    cm_props = defaultdict(list)
    cm_class = defaultdict(list)
    sample_class = defaultdict(lambda: defaultdict(str))

    # initialize flag variable
    num_amb_hit = 0
    is_fist_file = 1

    # read all files
    for dirpath, dirnames, filenames in os.walk(path_in):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # to get rid of ".DS_store"
                continue

            # full path to each blast output file
            full_path = os.path.join(dirpath, filename)

            if os.path.getsize(full_path) == 0:
                print("filesize 0")
                continue

            # get and save viral contigs
            df_one_sample, num_all_contigs, num_viral_contigs, num_viral_black, num_amb_hit_each = get_viral_hits(full_path, path_out, lineage_info, blacklist_path)
            num_amb_hit += num_amb_hit_each

            # store number of viral contigs per cm group
            cm_group = "external"
            cm_hits[cm_group].append(num_viral_black)
            cm_props[cm_group].append(num_viral_black / num_all_contigs)
    
            # infection 0 or 1 (use threshold)
            if num_viral_black >= infection_th:
                cm_class[cm_group].append("1")
            else:
                cm_class[cm_group].append("0")

            # concatnate results
            if is_fist_file == 1:
                df_all = df_one_sample.copy()
                is_fist_file = 0
            else:
                df_all = pd.concat([df_all, df_one_sample])

            # make info table per sample
            sample = full_path.split('/')[-1].split(".")[0].split("_")[0]
            if num_viral_black >= infection_th:
                sample_class[sample]["viral_contig"] = "1"
            else:
                sample_class[sample]["viral_contig"] = "0"
            sample_class[sample]["cm"] = cm_group

    print("Virus ambiguous hit:", num_amb_hit)
    print("=======")

    for k, v in cm_hits.items():
        print(k, np.mean(v))
    for k, v in cm_props.items():
        print(k, f"{np.mean(v):.2}")

    print("=======")
    return cm_hits, cm_props, cm_class, sample_class, df_all


def collect_from_tsv(tsv_file, output_file):
    """
    Read fasta path + contig IDs from a TSV and write selected sequences
    into a single multi-FASTA file.

    Parameters:
        tsv_file (str): Path to TSV file with columns [path, qseqid]
        output_file (str): Path to output fasta file
    """
    df = pd.read_table(tsv_file, sep="\t")
    with open(output_file, "w") as out_f:
        for fasta_file in sorted(list(set(df["path"].to_list()))):
            record_dict = SeqIO.index(fasta_file, "fasta")
            contigs = df.loc[df["path"] == fasta_file, "qseqid"].to_list()
            for contig in contigs:
                if contig in record_dict:
                    SeqIO.write(record_dict[contig], out_f, "fasta")
                else:
                    print(f"[Warning] {contig} not found in {fasta_file}")


def get_seq_from_fasta(fasta_file, contig_name):
    """
    Return the sequence string for a given contig name from a fasta file.

    Parameters:
        fasta_file (str): Path to fasta file
        contig_name (str): Contig ID (matches the part after '>' up to first space)

    Returns:
        str: Sequence as a string, or None if not found
    """
    record_dict = SeqIO.index(fasta_file, "fasta")
    if contig_name in record_dict:
        return record_dict[contig_name].seq
    else:
        return None


def asign_confusion_matrix(true_label, ml_label):
    res = np.nan
    if true_label == 1 and ml_label == 1:
        res = "TP"
    if true_label == 1 and ml_label == 0:
        res = "FN"
    if true_label == 0 and ml_label == 1:
        res = "FP"
    if true_label == 0 and ml_label == 0:
        res = "TN"
    if res != res:
        print("failed to assign cm group.")
    return res

# Taxonkit

In [4]:
# get all taxids
taxids = get_all_taxids(path_in, path_out + "taxids.txt")

34114 taxids in total


Run taxonkit (`Fig4_table_taxonkit.sh`) in `path_out` directory.

# Phage

In [5]:
# read taxonkit output
taxinfo = pd.read_table(path_out + "taxids_long.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# add taxonomy - {d};{K};{p};{c};{o};{f};{g};{s}
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# count NaN
print("#NaN:", taxinfo.isna().sum().sum())
# fill NaN
taxinfo = taxinfo.fillna("-")

# size
print("taxonkit output original size:", taxinfo.shape[0])

# find Caudoviricetes, phage, 
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    lineage = taxinfo.loc[idx, "lineage"]
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. phage sp
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# phage group order
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# size
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# save table
taxinfo.to_csv(path_out + "taxids_phage.tsv", sep="\t")

# save phage list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# show
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 18
taxonkit output original size: 34114
phage order: ['Crassvirales', 'unclassified Caudoviricetes order', 'unclassified Viruses order']
Caudoviricetes: 261
phage: 108
 * not Caudoviricetes: 3
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae
15512,1892901,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,ssRNA phage DC,1,0,1,0,0,0
23991,278008,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured phage,1,0,1,0,0,0
26413,38018,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Bacteriophage sp.,1,0,1,0,0,0


# Add taxonomy

In [6]:
# read taxonkit output
taxid_in = path_out + "taxids_lineage.txt"
lineage_info = pd.read_table(taxid_in, header=None)
lineage_info.columns = ["taxid", "domain", "fgs"]

# check NaN
print("#NaN beginning:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# add family, genus, speceis column
lineage_info["taxid"] = lineage_info["taxid"].astype(str)
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)  # if nan
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)  # if nan
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)  # if nan

# replace domain with phage for phage group
phage_ids = []
with open(path_out + "phage_list.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# check NaN
print("#NaN end:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# fill NaN
lineage_info = lineage_info.fillna("-")

print(f"{lineage_info.shape[0]} taxids in total\n")

# show example hit for each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# show
lineage_info

#NaN beginning: 4
        taxid domain  fgs
3288  1230063    NaN  NaN
3291  1230068    NaN  NaN

#NaN end: 10
        domain  fgs family genus species
taxid                                   
1230063    NaN  NaN    NaN   NaN     NaN
1230068    NaN  NaN    NaN   NaN     NaN

34114 taxids in total

All domain: {'Eukaryota', 'unclassified unclassified entries domain', 'unclassified other entries domain', 'Archaea', '-', 'Bacteria', 'unclassified Viruses domain', 'phage_group'} 

##### Eukaryota ####
size: 3427
            domain                      species
taxid                                          
100035   Eukaryota  Massariosphaeria phaeospora
1000413  Eukaryota              Acer yangbiense
100182   Eukaryota            Lepus granatensis

##### unclassified unclassified entries domain ####
size: 2
                                          domain              species
taxid                                                                
155900  unclassified unclassified entries doma

,domain,fgs,family,genus,species
taxid,,,,,
1000289,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000293,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000294,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000295,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000296,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
...,...,...,...,...,...
999810,Eukaryota,Sclerotiniaceae;Botrytis;Botrytis cinerea,Sclerotiniaceae,Botrytis,Botrytis cinerea
999849,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
999883,unclassified Viruses domain,Marseilleviridae;unclassified Marseilleviridae...,Marseilleviridae,unclassified Marseilleviridae genus,Lausannevirus


# Extract viral tophit

In [7]:
# extract virus tophit
infection_threshod = 1
cm_hits, cm_props, cm_class, sample_class, df_all = count_viral_contigs(path_in, path_out, lineage_info, blacklist_path, infection_threshod)

# viral contig table
df = df_all.copy().reset_index()
# check sample names
for i in df.index:
    qseqid = df.loc[i, "qseqid"]
    if type(qseqid)==int:
        continue
        print(df.loc[i, :])
    else:
        if qseqid.split("_")[0]!=df.loc[i, "file_name"]:
           print(df.loc[i, :])
print("done")
df.columns = df.columns.map(lambda x: x.replace("file_name", "sample"))

# cm group
sampledf = pd.DataFrame(sample_class).T
df = df.merge(sampledf, left_on = "sample", right_index=True, how="left")

# add count of viral contigs per sample
virus_hit_count = pd.DataFrame(df.value_counts("sample"))
df = df.merge(virus_hit_count, on="sample", how="left")
df = df.sort_values("sample")
df = df.sort_values("count", ascending=False)

# count NaN
print("#NaN:", df.isna().sum().sum())

# save
df.to_csv(path_out + "virus_all.txt", sep="\t")

Virus ambiguous hit: 9006
external 5.720734506503443
external 0.0094
done
#NaN: 0


# BLASTn

In [8]:
## blastn
# create viral contig + path table
df = pd.read_table(path_out + "virus_all.txt", index_col=0)
df = df.drop("viral_contig", axis=1)
df_fasta = df[["qseqid", "cm", "sample"]].copy()
for idx in df_fasta.index:
    df_fasta.loc[idx, "path"] = f"{path_in_fasta}{df_fasta.loc[idx, "sample"]}.viral_hit_contigs.fa"
df_fasta = df_fasta[["path", "qseqid"]]
# save table
df_fasta.to_csv(path_out + "virus_contig_path.txt", sep="\t", index=False)
# create fasta file of viral contigs
collect_from_tsv(path_out + "virus_contig_path.txt", path_out + "virus_hit_all_contigs.fasta")

Run BLASTn (script: `Fig4_table_blastn.sh`, output file name: `path_in_blastn`)

# Taxonkit for blastn

In [9]:
## taxonkit
# path for taxonkit result
takonkit_path_2 = path_out + "taxids_blastn.txt"

# initialize set
taxids = set()

# read blastn result
col_name= [
    "qseqid", "sseqid", "pident", "length", 
    "mismatch", "gapopen", "qstart", "qend", 
    "sstart", "send", "evalue", "bitscore", 
    "qframe", "staxids", "stitle", "qlen", 
    "slen", "qcovs", "qcovhsp"
]

blastn = pd.read_table(path_in_blastn, header=None, dtype={13:str})  # staxids as string
blastn.columns = col_name
print("blastn output original size:", blastn.shape)
print("#NaN:", blastn.isna().sum().sum())

# sort by bitscore
blastn_sorted = blastn.sort_values("bitscore", ascending=False)
# delete duplicates
blastn_uniq = blastn_sorted.groupby("qseqid").first()
print("blastn output size after choosing tophits:", blastn_uniq.shape)

# get taxid
for line_taxid in blastn_uniq["staxids"].to_list():
    if (line_taxid is None) or (line_taxid!=line_taxid):
        continue
    raw_taxids = line_taxid.split(";")
    for i in range(len(raw_taxids)):
        taxids.add(raw_taxids[i])

# write taxid per line
with open(takonkit_path_2, "w") as f:
    for taxid in taxids:
        f.write(taxid + "\n")

print(f"{len(taxids)} taxids in total")

# show
blastn_uniq.head(10)

blastn output original size: (6914676, 19)
#NaN: 0
blastn output size after choosing tophits: (20808, 18)
1832 taxids in total


,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qframe,staxids,stitle,qlen,slen,qcovs,qcovhsp
qseqid,,,,,,,,,,,,,,,,,,
102179,gi|2196303737|gb|MZ679293.1|,91.579,285,24,0,1,285,6037,5753,3.980000e-105,394.0,1,1930506,MAG: Sicinivirus sp. isolate R22-k141_395554 p...,285,6092,100,100
1042341,gi|2171934023|gb|OM079140.1|,100.000,213,0,0,1,213,8015,7803,2.830000e-105,394.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,213,29799,100,100
1060688,gi|2493070242|gb|OQ852598.1|,99.892,928,1,0,1,928,22073,23000,0.000000e+00,1709.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,928,29706,100,100
1088902,gi|2240251053|gb|MT978647.1|,77.665,197,42,2,19,214,299,104,1.920000e-22,119.0,1,10788,Canine parvovirus strain CPV-2c/Myanmar/RS-45/...,214,1746,92,92
113348,gi|2196303729|gb|MZ679278.1|,96.495,428,15,0,1,428,3601,4028,0.000000e+00,708.0,1,1930506,MAG: Sicinivirus sp. isolate 363R-k141_188215 ...,428,5304,100,100
1154076,gi|2305169510|gb|OP468871.1|,100.000,478,0,0,1,478,24866,24389,0.000000e+00,883.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,478,29718,100,100
117398,gi|2918479373|gb|PV093045.1|,91.541,863,73,0,1,863,6413,7275,0.000000e+00,1190.0,1,1930506,"Sicinivirus sp. strain WHM13, partial genome",863,9854,100,100
1190696,gi|2112847088|emb|OU815371.1|,100.000,407,0,0,1,407,13262,12856,0.000000e+00,752.0,1,2697049,Severe acute respiratory syndrome coronavirus ...,407,29899,100,100
119917,gi|408684319|dbj|AB753441.1|,100.000,1064,0,0,1,1064,1608,2671,0.000000e+00,1965.0,1,3052731,"Respirovirus muris P, N genes for phosphoprote...",1064,3431,100,100


Run `Fig4_table_taxonkit_blastn.sh` in `path_out` directory.

# Phage for blastn

In [10]:
## phage list
# read taxonkit output
taxinfo = pd.read_table(path_out + "taxids_long_blastn.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# add taxonomy - {d};{K};{p};{c};{o};{f};{g};{s}
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# count NaN
print("#NaN:", taxinfo.isna().sum().sum())
# fill NaN
taxinfo = taxinfo.fillna("-")
# size
print("taxonkit result size:", taxinfo.shape[0])

# find Caudoviricetes, phage, Inoviridae, Lavidaviridae, Microviridae
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    lineage = taxinfo.loc[idx, "lineage"]
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. phage sp
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# phage group order
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# size
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# save table
taxinfo.to_csv(path_out + "taxids_phage.tsv", sep="\t")

# save phage list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list_blastn.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# show
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 9
taxonkit result size: 1832
phage order: []
Caudoviricetes: 1
phage: 0
 * not Caudoviricetes: 0
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae


In [11]:
## add lineage info to taxid list
# input path
taxid_in = path_out + "taxids_lineage_blastn.txt"

# read lineage table
lineage_info = pd.read_table(taxid_in, header=None, dtype={0:str})
lineage_info.columns = ["taxid", "domain", "fgs"]

# supplyment 3434951 (deleted))
lineage_info.loc[lineage_info["taxid"]=="3434951", "domain"] = "unclassified Viruses domain"
lineage_info.loc[lineage_info["taxid"]=="3434951", "fgs"] = "Picobirnaviridae;Orthopicobirnavirus;unclassified Orthopicobirnavirus"

# check NaN
print("#NaN:", lineage_info.isna().sum().sum())

# add family, genus, speceis columns
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)


# replace domain with phage for phage group
phage_ids = []
with open(path_out + "phage_list_blastn.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# fill NaN
lineage_info = lineage_info.fillna("-")

# size
print(f"{lineage_info.shape[0]} taxids in total\n")

# show example hit for each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# show
lineage_info

#NaN: 0
1832 taxids in total

All domain: {'Eukaryota', 'unclassified other entries domain', 'Bacteria', 'unclassified Viruses domain', 'phage_group'} 

##### Eukaryota ####
size: 37
           domain              species
taxid                                 
42410   Eukaryota  Peromyscus eremicus
885580  Eukaryota   Fukomys damarensis
94328   Eukaryota  Zingiber officinale

##### unclassified other entries domain ####
size: 9
                                    domain  \
taxid                                        
3390493  unclassified other entries domain   
161364   unclassified other entries domain   
167860   unclassified other entries domain   

                                                  species  
taxid                                                      
3390493  Cloning vector CMVn-OC43-mClover-Ribo-BGH-YCpBAC  
161364                             Cloning vector pAAV-RC  
167860                         Cloning vector pHCMV-VSV-G  

##### Bacteria ####
size: 1
        

,domain,fgs,family,genus,species
taxid,,,,,
664134,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
3349264,unclassified Viruses domain,Picobirnaviridae;Picobirnavirus;Rattus norvegi...,Picobirnaviridae,Picobirnavirus,Rattus norvegicus associated picobirnavirus 33
713104,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
3139435,unclassified Viruses domain,Paramyxoviridae;Morbillivirus;Morbillivirus canis,Paramyxoviridae,Morbillivirus,Morbillivirus canis
1841013,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
...,...,...,...,...,...
1731326,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
311710,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1840878,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae


In [12]:
## merge blastn result and taxomoy info
# read blastn result
blastn_tax = blastn_uniq.copy()
blastn_tax["staxids_lst"] = blastn_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
blastn_tax["name_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
blastn_tax["group_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

# add taxonomy lineage
num_ambiguous_hit = 0
for idx in blastn_uniq.index:
    groups = blastn_tax.loc[idx, "group_lst"]
    names = blastn_tax.loc[idx, "name_lst"]
    # count hits with taxids with multiple groups
    if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
        num_ambiguous_hit += 1
        print("#### Different root ####")
        print(groups)
        print(names)
        print()
print("Num ambiguou hits:", num_ambiguous_hit)

# add family, genus, species
blastn_tax["taxid"] = blastn_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  # added on 2025/09/01
blastn_tax["family"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "family"] if((x is not None) and (x==x)) else x)
blastn_tax["genus"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "genus"] if((x is not None) and (x==x)) else x)
blastn_tax["species"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "species"] if((x is not None) and (x==x)) else x)

# check NaN
print("#NaN:", blastn_tax.isna().sum().sum())

# add tag if list has virus - virus = yes when all classification are virus
blastn_tax["is_virus"] = blastn_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

# edit columns name
blastn_tax.columns = blastn_tax.columns.map(lambda x: x + "_blastn")  # add "_blastn" to column names

# save
blastn_tax.to_csv(path_out + "blastn_result_top_hit.tsv", sep="\t")

# show
print("Size after adding taxonomy:", blastn_tax.shape)
blastn_tax.head(10)

Num ambiguou hits: 0
#NaN: 0
Size after adding taxonomy: (20808, 26)


,sseqid_blastn,pident_blastn,length_blastn,mismatch_blastn,gapopen_blastn,qstart_blastn,qend_blastn,sstart_blastn,send_blastn,evalue_blastn,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
qseqid,,,,,,,,,,,,,,,,,,,,,
102179,gi|2196303737|gb|MZ679293.1|,91.579,285,24,0,1,285,6037,5753,3.980000e-105,...,100,100,[1930506],[Picornaviridae;Sicinivirus;Sicinivirus sp.],[unclassified Viruses domain],1930506,Picornaviridae,Sicinivirus,Sicinivirus sp.,1
1042341,gi|2171934023|gb|OM079140.1|,100.000,213,0,0,1,213,8015,7803,2.830000e-105,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
1060688,gi|2493070242|gb|OQ852598.1|,99.892,928,1,0,1,928,22073,23000,0.000000e+00,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
1088902,gi|2240251053|gb|MT978647.1|,77.665,197,42,2,19,214,299,104,1.920000e-22,...,92,92,[10788],[Parvoviridae;Protoparvovirus;Protoparvovirus ...,[unclassified Viruses domain],10788,Parvoviridae,Protoparvovirus,Protoparvovirus carnivoran1,1
113348,gi|2196303729|gb|MZ679278.1|,96.495,428,15,0,1,428,3601,4028,0.000000e+00,...,100,100,[1930506],[Picornaviridae;Sicinivirus;Sicinivirus sp.],[unclassified Viruses domain],1930506,Picornaviridae,Sicinivirus,Sicinivirus sp.,1
1154076,gi|2305169510|gb|OP468871.1|,100.000,478,0,0,1,478,24866,24389,0.000000e+00,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
117398,gi|2918479373|gb|PV093045.1|,91.541,863,73,0,1,863,6413,7275,0.000000e+00,...,100,100,[1930506],[Picornaviridae;Sicinivirus;Sicinivirus sp.],[unclassified Viruses domain],1930506,Picornaviridae,Sicinivirus,Sicinivirus sp.,1
1190696,gi|2112847088|emb|OU815371.1|,100.000,407,0,0,1,407,13262,12856,0.000000e+00,...,100,100,[2697049],[Coronaviridae;Betacoronavirus;Betacoronavirus...,[unclassified Viruses domain],2697049,Coronaviridae,Betacoronavirus,Betacoronavirus pandemicum,1
119917,gi|408684319|dbj|AB753441.1|,100.000,1064,0,0,1,1064,1608,2671,0.000000e+00,...,100,100,[3052731],[Paramyxoviridae;Respirovirus;Respirovirus muris],[unclassified Viruses domain],3052731,Paramyxoviridae,Respirovirus,Respirovirus muris,1


# Save

In [13]:
## random sampling
# read sample info table
sample_info = pd.read_table(sradata_path)
sample_info_analyzed = sample_info.loc[sample_info["diamond_blastx"] == 1, :].copy()
# sample 1000
# define sample size
sampe_size = 1000 - sample_info_analyzed.loc[((sample_info_analyzed["Negative"]==3)|(sample_info_analyzed["Negative"]==4)),:].shape[0]
random.seed(42) 
subset_lst = random.sample(sample_info_analyzed.loc[sample_info_analyzed["Negative"]==5, "ID"].to_list(), sampe_size)
sample_info_analyzed["random_sampling"] = sample_info_analyzed.apply(
    lambda x: 1 if x["Negative"] != 5 else (1 if x["ID"] in subset_lst else 0),
    axis=1
)
# add logan result
sra_meta = pd.read_table(srameta_path)
sra_meta.columns = sra_meta.columns.map(lambda x: x + "_host" if x in ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'] else x)
# merge
combined = sample_info_analyzed.merge(sra_meta, on="ID", how="left")
# ML label
combined["ML_label"] = combined["Positive"].apply(lambda x: 1 if x >= 3 else 0)
# True label
combined["True_label"] = combined["ISG_level"].apply(lambda x: 1 if x in ["high", "both"] else (0 if x in ["Negative", "low"] else np.nan))
# cm
combined["cm"] = combined.apply(lambda x: asign_confusion_matrix(x["True_label"], x["ML_label"]), axis=1)
# save
combined.to_csv(path_out + "cm_assignment.tsv", sep="\t")
combined.head(3)

,ID,Negative,Positive,diamond_blastx,random_sampling,Host_species,BioProject_ID,species_host,tax_id,kingdom_host,...,class_host,order_host,family_host,genus_host,nucleotide_DNA,nucleotide_RNA,ISG_level,ML_label,True_label,cm
0,ERR11219020,0,5,1,1,Sus_scrofa,PRJEB61259,Sus_scrofa,9823,Eukaryota,...,Mammalia,Artiodactyla,Suidae,Sus,Positive,Negative,high,1,1,TP
1,ERR11219054,0,5,1,1,Sus_scrofa,PRJEB61259,Sus_scrofa,9823,Eukaryota,...,Mammalia,Artiodactyla,Suidae,Sus,Negative,Positive,high,1,1,TP
2,ERR11630589,2,3,1,1,Gallus_gallus,PRJEB63475,Gallus_gallus,9031,Eukaryota,...,Aves,Galliformes,Phasianidae,Gallus,Negative,Negative,Negative,1,0,FP


In [ ]:
## merge blastn result and blastx result
# read blastx result
df_all = pd.read_table(path_out + "virus_all.txt", index_col=0)

# read blastn result
df_blastn = pd.read_table(path_out + "blastn_result_top_hit.tsv")
print("df_blastn shape:", df_blastn.shape)

# merge blastn and blastx
df_all_blastn = df_all.merge(df_blastn, on="qseqid", how="left")
print("df_all_blastn shape:", df_all_blastn.shape)

# random sampling
cm_info = pd.read_table(path_out + "cm_assignment.tsv", index_col=0)
df_all_blastn = df_all_blastn.drop("cm", axis=1)
df_all_blastn = df_all_blastn.merge(cm_info[["ID", "random_sampling", "cm", "ML_label"]], left_on="sample", right_on="ID", how="left")
df_all_sampled = df_all_blastn.loc[df_all_blastn["random_sampling"]==1, :].copy()
print("size of df_all_sampled:", df_all_sampled.shape)

# add query seq
df_all_sampled["query_seq"] = df_all_sampled.apply(lambda row: get_seq_from_fasta(f"{path_in_fasta}{row["sample"]}.viral_hit_contigs.fa", row["qseqid"]), axis=1)
df_all_sampled["qseqid_formatted"] = df_all_sampled.apply(lambda x: x["sample"] + "_" + x["qseqid"] if not(x["qseqid"].startswith("E") or x["qseqid"].startswith("S")) else x["qseqid"], axis=1)

# save
df_all_sampled.to_csv(path_out + "virus_all.blastn.randomsampled.tsv", sep="\t")
# show
df_all_sampled

df_blastn shape: (20808, 27)
df_all_blastn shape: (22431, 57)
size of df_all_sampled: (22414, 60)
